# Hands-on: Simple RAG from Scratch!

Disiapkan oleh Tegar Arifin Prasetyo, S.Si., M.Si

# 1. Setup & Install Library

Sebelum menjalankan notebook ini, pastikan runtime sudah diset ke **GPU (T4)**: Runtime --> Change runtime type --> T4 GPU --> Save

### Penjelasan sel: Mengecek perangkat yang digunakan

Sel ini memeriksa apakah GPU tersedia. Jika tersedia, variabel `DEVICE` berisi `cuda`. Jika tidak, nilainya `cpu`. Variabel ini dipakai saat memuat model pada langkah 7. Sel ini tidak menampilkan output karena tidak ada perintah `print()`.


In [12]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

### Penjelasan sel: Memasang library

Sel ini memasang library yang dibutuhkan untuk membaca PDF, memotong teks, membuat embedding, mencari teks, dan menjalankan model. Tunggu sampai proses selesai sebelum lanjut. Opsi `-q` mengurangi tulisan proses instalasi yang muncul di layar.


In [4]:
!pip install -q \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    accelerate

# 2. Setup HuggingFace Token

HuggingFace Token dibutuhkan untuk mengakses model dan mendownload weights dari **HuggingFace Hub**.

**Cara mendapatkan token gratis:**
1. Buka https://huggingface.co/settings/tokens
2. Klik **"+ Create new token"**
3. Pilih token type: **Read**
4. Buat nama token, lalu klik **Create token**
5. Copy dan simpan tokennya

**Cara setup HuggingFace Token di Google Colab dengan Secrets:**
> Lebih aman karena token tidak terlihat secara langsung di notebook
1. Klik ikon 🔑 di sidebar kiri Colab
2. Klik **"Add new secret"**
3. Name: `HF_TOKEN`, Value: paste token dari HuggingFace
4. Aktifkan toggle **"Notebook access"**

### Penjelasan sel: Mengambil token HuggingFace

Sel ini mengambil token dari Secrets Google Colab dengan nama `HF_TOKEN`, lalu memasukkannya ke pengaturan lingkungan notebook. Buat secret tersebut dan aktifkan Notebook access sebelum menjalankan sel. Output hanya menampilkan beberapa karakter awal token, bukan token lengkap.


In [5]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

# Set sebagai environment variable agar otomatis dikenali transformers & HF Hub
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
print(f"HF Token prefix: {HF_TOKEN[:8]}...")

HF Token prefix: hf_zrMXr...


# 3. Upload & Load Dokumen PDF

Tahap pertama pipeline RAG: membaca isi dokumen PDF dan mengubahnya menjadi
objek `Document` yang bisa diproses LangChain.

> Format lain (`.txt`, `.docx`, `.csv`) juga didukung LangChain, namun membutuhkan loader yang berbeda.

### Penjelasan sel: Mengunggah dan membaca PDF

Setelah sel dijalankan, pilih satu file PDF. Kode mengambil teks dari setiap halaman dan menyimpan teks beserta nomor halaman dan nama file ke dalam `documents`. Hasilnya menampilkan jumlah halaman dan contoh teks. Gunakan PDF yang teksnya bisa disalin; kode ini belum membaca gambar dengan OCR. Pastikan jumlah halaman dengan teks lebih dari 0 agar `documents[0]` dapat dibaca. Nomor halaman pada metadata dimulai dari 0.


In [6]:
from google.colab import files
from pypdf import PdfReader
from langchain_core.documents import Document

# Upload PDF
print("Pilih file PDF ...")
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f"File PDF berhasil diupload: {pdf_filename}")

# Load & ekstrak PDF
# Setiap halaman dijadikan satu objek Document
# dengan metadata page number & nama file sebagai sumbernya
reader = PdfReader(pdf_filename)
documents = []
for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():
        documents.append(Document(
            page_content=text,
            metadata={"page": page_num, "source": pdf_filename}
        ))

# Preview PDF
print(f"\nFile PDF berhasil diload!")
print(f"   Total halaman       : {len(reader.pages)}")
print(f"   Halaman dengan teks : {len(documents)}")
print(f"\nContoh isi halaman pertama (200 karakter pertama):")
print(f" '{documents[0].page_content[:200]}...'")

Pilih file PDF ...


Saving Accountability_and_Workplace_Relationships_compressed.pdf to Accountability_and_Workplace_Relationships_compressed.pdf
File PDF berhasil diupload: Accountability_and_Workplace_Relationships_compressed.pdf

File PDF berhasil diload!
   Total halaman       : 19
   Halaman dengan teks : 19

Contoh isi halaman pertama (200 karakter pertama):
 'ACCOUNTABILITY ANDACCOUNTABILITY AND
WORKPLACE RELATIONSHIPSWORKPLACE RELATIONSHIPS
ACCOUNTABILITY AND
WORKPLACE RELATIONSHIPS
Hernawati Samosir
...'


# 4. Chucking

Dokumen yang panjang tidak bisa langsung diproses LLM karena adanya batasan
**context window** (jumlah token maksimum yang bisa diproses sekaligus).
Chunking memecah dokumen menjadi potongan-potongan kecil yang lebih mudah dicari
dan dimasukkan ke dalam prompt.

**Parameter chunking:**
- `chunk_size` — jumlah karakter per chunk
- `chunk_overlap` — jumlah karakter yang overlap antar chunk

**Kenapa perlu parameter overlap?**
Tanpa parameter overlap, informasi yang berada di batas antar chunk bisa terpotong dan hilang. Parameter Overlap memastikan koneksi antar kalimat tetap terjaga.


### Penjelasan sel: Memotong teks menjadi bagian kecil

Sel ini membagi teks PDF menjadi potongan kecil yang disebut chunk. Panjang maksimum setiap chunk adalah 500 karakter, dengan overlap hingga 50 karakter agar konteks antarpotongan tetap terhubung. Hasilnya disimpan dalam `chunks`. Periksa jumlah chunk dan contoh teks yang muncul setelah sel dijalankan.


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Parameter chunking
# chunk_size    : panjang maksimum tiap chunk (dalam karakter)
# chunk_overlap : jumlah karakter yang overlap antar chunk
# separators    : urutan pemisah yang dicoba.dari yang paling diutamakan
#                 ke yang paling akhir (karakter tunggal / kosong)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chuck dokumen
chunks = text_splitter.split_documents(documents)

# Preview hasil Chucking
print(f"Hasil chunking:")
print(f"   Total halaman asli : {len(documents)}")
print(f"   Total chunks       : {len(chunks)}")
print(f"   Rata-rata panjang  : {sum(len(c.page_content) for c in chunks) // len(chunks)} karakter")
print(f"\nContoh chunk pertama:")
print(f"   Metadata : {chunks[0].metadata}")
print(f"   Konten   : '{chunks[0].page_content[:200]}...'")

Hasil chunking:
   Total halaman asli : 19
   Total chunks       : 32
   Rata-rata panjang  : 330 karakter

Contoh chunk pertama:
   Metadata : {'page': 0, 'source': 'Accountability_and_Workplace_Relationships_compressed.pdf'}
   Konten   : 'ACCOUNTABILITY ANDACCOUNTABILITY AND
WORKPLACE RELATIONSHIPSWORKPLACE RELATIONSHIPS
ACCOUNTABILITY AND
WORKPLACE RELATIONSHIPS
Hernawati Samosir...'


# 5. Embedding & Vector Database

Tahap ini mengubah setiap chunk teks menjadi **vektor embedding** yang merepresentasikan makna teks tersebut, lalu menyimpannya ke **FAISS** sebagai vector database yang bisa dicari secara semantik.

Contoh: kata yang maknanya mirip --> vektornya berdekatan di ruang vektor
```
"kucing"  → [0.2, 0.8, 0.1, ...]
"anjing"  → [0.3, 0.7, 0.2, ...]  <-- dekat dengan kucing
"laptop"  → [0.9, 0.1, 0.7, ...]  <-- jauh dari kucing
```

**Dua komponen utama:**
1. **Embedding model** (`all-MiniLM-L6-v2`) https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
  > Digunakan untuk mengubah teks menjadi vektor. Model kecil (~80MB), jalan di CPU, dioptimalkan untuk semantic search. Berbeda dengan model LLM, embedding model **tidak generate teks**, hanya menghasilkan representasi numerik dari makna sebuah teks.

2. **FAISS** (Facebook AI Similarity Search) https://github.com/facebookresearch/faiss
  > Digunakan untuk menyimpan & mencari vektor. Mencari chunk yang paling relevan berdasarkan **kemiripan makna**, bukan kesamaan kata. Contoh: query *"cara daftar beasiswa"* bisa menemukan chunk yang berisi *"prosedur pendaftaran"* meskipun kata-katanya berbeda.

### Penjelasan sel: Membuat embedding dan pencarian teks

Sel ini memuat model embedding `paraphrase-multilingual-MiniLM-L12-v2`. Model mengubah teks menjadi deretan angka yang mewakili maknanya. Angka tersebut disimpan dalam indeks FAISS agar potongan teks yang relevan bisa dicari. Fungsi `retrieve()` mengambil beberapa chunk yang paling mirip dengan pertanyaan. Bagian akhir mencoba pencarian dan menampilkan contoh hasilnya. Sel ini memerlukan `chunks` dari langkah sebelumnya.


In [8]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Download & Load embedding model
# Model ini berjalan lokal di Colab (CPU), bukan via API
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# Model lain bisa dicari di https://huggingface.co/sentence-transformers
# Contoh:
# sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# sentence-transformers/all-MiniLM-L6-v2
# BAAI/bge-m3

# Buat embedding untuk semua chunks
# Setiap chunk diubah menjadi vektor 384 dimensi
# normalize_embeddings=True --> memungkinkan cosine similarity via inner product
print(f"\nMembuat embedding untuk {len(chunks)} chunks...")
chunk_texts = [c.page_content for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Buat FAISS index
dim = chunk_embeddings.shape[1]          # dimensi vektor (all-MiniLM-L6-v2)
index = faiss.IndexFlatIP(dim)           # Inner Product = cosine similarity (karena normalized)
index.add(np.array(chunk_embeddings))    # masukkan semua embedding ke index

print(f"\nVector database berhasil dibuat!")
print(f"   Dimensi vektor  : {dim}")
print(f"   Total chunks    : {index.ntotal}")

# Helper: fungsi retrieval
# Mengubah query teks --> vektor, lalu mencari k chunks
# yang paling mirip secara semantik di FAISS index
def retrieve(query: str, k: int = 3):
    """Cari k chunks paling relevan untuk query."""
    query_vec = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]

# Test retrieval
print(f"\nTest retrieval — cari 2 chunk paling relevan:")
test_results = retrieve("What is this document about?", k=2)
for i, doc in enumerate(test_results):
    print(f"\n   Result {i+1} (halaman {doc.metadata.get('page', '?')}):")
    print(f"   '{doc.page_content[:150]}...'")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Membuat embedding untuk 32 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Vector database berhasil dibuat!
   Dimensi vektor  : 384
   Total chunks    : 32

Test retrieval — cari 2 chunk paling relevan:

   Result 1 (halaman 1):
   'issues
LEARNING OUTCOMESLEARNING OUTCOMESLEARNING OUTCOMES
Institut Teknologi Del 02...'

   Result 2 (halaman 17):
   'Text
Activity #6:
Institut Teknologi Del 03
Text Activity #6: Accountability Measures (Activity 6-
4)
Identify tasks required to complete major goals ...'


# 6. Konfigurasi Model LLM

Bagian ini menentukan model LLM yang akan digunakan untuk **generate jawaban**.

> 💡 Model akan didownload dari HuggingFace Hub.
> Download hanya dilakukan **sekali per session**. Jika runtime tidak di-restart, model tetap tersedia.

**Syarat model yang bisa dipakai untuk RAG:**
1. Tersedia di HuggingFace Hub https://huggingface.co/models. Model di luar HuggingFace bisa dipakai tapi perlu di-load manual dari path lokal.
2. Versi **Instruct** atau **Chat**, bukan base model. Base model hanya melanjutkan teks, bukan menjawab pertanyaan. Sedangkan versi Instruct/Chat sudah dilatih khusus untuk mengikuti instruksi sehingga cocok untuk pipeline RAG.
3. Ukuran sesuai VRAM (memori GPU) yang tersedia. Jika ukuran model melebihi VRAM, loading model akan gagal.


### Penjelasan sel: Memilih model untuk menjawab pertanyaan

Sel ini menentukan nama model LLM yang akan dipakai. Notebook menggunakan `Qwen/Qwen2.5-1.5B-Instruct`. Sel ini hanya menyimpan nama model ke `MODEL_ID`; proses mengunduh dan memuat model dilakukan pada sel berikutnya. Untuk percobaan awal, nama model tidak perlu diganti.


In [9]:
# Ganti MODEL_ID dengan model LLM yang diinginkan
# Contoh: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 7. Load Model & Bangun RAG Chain

Tahap ini melakukan dua hal sekaligus:
1. **Download & load LLM** dari HuggingFace Hub ke memori GPU
2. **Bangun RAG chain** yang menghubungkan semua komponen

### Penjelasan sel: Memuat model dan menghubungkan alur RAG

Sel ini mengunduh dan memuat LLM, lalu membuat alur untuk menjawab pertanyaan. Pertanyaan dipakai untuk mencari tiga chunk yang relevan. Teks hasil pencarian dimasukkan ke prompt bersama pertanyaan, lalu model membuat jawaban. Prompt meminta model menjawab berdasarkan konteks yang diberikan dan mengaku jika informasinya tidak cukup; hasilnya tetap perlu diperiksa. Pastikan `DEVICE`, `MODEL_ID`, dan fungsi `retrieve()` sudah tersedia. Tunggu sampai model selesai dimuat. Batas jawaban diatur pada `max_new_tokens=512`.


In [10]:
import torch
import transformers
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# Download & Load Model LLM
# Model di-download dari HuggingFace Hub ke Colab session
# device_map="auto" --> otomatis load ke GPU jika tersedia, fallback ke CPU
pipe = transformers.pipeline(
    "text-generation",
    model=MODEL_ID,
    model_kwargs={
        "torch_dtype": torch.bfloat16 if DEVICE == "cuda" else torch.float32
    },
    device_map="auto",
)

print(f"Model berhasil di-load!")
print(f"Model ID : {MODEL_ID}")

# Fungsi generate jawaban
# Menerima konteks + pertanyaan, mengembalikan jawaban dari LLM
def call_llm(prompt_text: str) -> str:
    """Generate jawaban dari model lokal."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer based ONLY on the provided context. Be concise."},
        {"role": "user",   "content": prompt_text},
    ]
    # max_new_tokens --> Jumlah maksimum token yang boleh di-generate model
    # temperature --> Mengontrol "kreativitas" / keacakan output
    # do_sample --> True agar temperature berlaku; False = greedy decoding (selalu pilih token probabilitas tertinggi)
    outputs = pipe(
        messages,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
    )
    # Ambil hanya teks jawaban terakhir (bukan history)
    return outputs[0]["generated_text"][-1]["content"].strip()

# Prompt Augmentation
# Instruksi (prompt) eksplisit agar model LLM hanya menjawab dari konteks
# {context} dan {question} diisi otomatis oleh RAG chain
RAG_PROMPT = """Answer the question based ONLY on the context below.
If not found in context, say \"I don't have enough information.\"

Context:
{context}

Question: {question}"""

prompt = PromptTemplate(
    template=RAG_PROMPT,
    input_variables=["context", "question"]
)

# Helper: retrieve + format konteks
# Menggabungkan hasil retrieve() menjadi satu blok teks
# dengan label halaman sebagai referensi sumber
def format_docs(question: str) -> str:
    docs = retrieve(question, k=3)
    return "\n\n".join([
        f"[Halaman {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    ])

# Bangun RAG Chain Pipeline
# Menghubungkan semua komponen menjadi satu pipeline
def rag_pipeline(question: str) -> str:
    context = format_docs(question)
    filled_prompt = prompt.format(context=context, question=question)
    return call_llm(filled_prompt)

rag_chain = RunnableLambda(rag_pipeline)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model berhasil di-load!
Model ID : Qwen/Qwen2.5-1.5B-Instruct


# 8. Test RAG Chain

Sebelum membungkus dengan UI, uji RAG chain langsung di console.
Output verbose menampilkan chunks yang ditemukan retriever — berguna untuk
memastikan retrieval bekerja dengan benar sebelum lanjut ke Gradio.

**Ganti pertanyaan** di `test_questions` sesuai isi PDF yang dipakai.

### Penjelasan sel: Membuat fungsi pengujian chatbot

Sel ini membuat fungsi `ask()` agar pengujian lebih mudah. Saat `verbose=True`, fungsi menampilkan pertanyaan, contoh chunk hasil pencarian, dan jawaban model. Saat `verbose=False`, fungsi hanya mengembalikan jawabannya. Sel ini baru membuat fungsi, jadi belum menghasilkan jawaban sampai fungsi dipanggil.


In [13]:
# Fungsi untuk bertanya ke RAG chain.
# verbose=True  --> tampilkan chunks yang ditemukan + jawaban
# verbose=False --> hanya kembalikan jawaban (dipakai untuk inference di Gradio)
def ask(question: str, verbose: bool = True):
    """
    Fungsi utama untuk bertanya ke chatbot.
    Set verbose=True untuk lihat chunks yang diambil.
    """
    if verbose:
        print(f"Pertanyaan: {question}")

        # Tampilkan chunks yang ditemukan retriever
        # Berguna untuk debug. Apakah retrieval sudah relevan?
        print("\nChunks yang ditemukan (retrieval):")
        retrieved = retrieve(question, k=3)   # pakai fungsi retrieve()
        for i, doc in enumerate(retrieved):
            print(f"  [{i+1}] Halaman {doc.metadata.get('page','?')}: '{doc.page_content[:100]}...'")
        print()

    answer = rag_chain.invoke(question)
    answer = answer.strip()

    if verbose:
        print(f"Jawaban:")
        print(f"   {answer}")
        print("-" * 60)

    return answer

### Penjelasan sel: Menjalankan pertanyaan uji

Sel ini mengirim tujuh pertanyaan tentang PDF Accountability secara bergantian ke fungsi `ask()`. Periksa apakah hasil pencarian sesuai topik dan jawabannya cocok dengan isi PDF. Jika dokumen diganti, ubah daftar `test_questions` agar sesuai dokumen baru. Pengujian ini membantu mengecek hasil, tetapi belum membuktikan semua pertanyaan akan dijawab dengan benar.


In [14]:
# Ganti pertanyaan berikut sesuai isi PDF
test_questions = [
    "What does accountability mean in the workplace?",
    "Why should you avoid speaking poorly of others when networking?",
    "What additional activities could Christine do to demonstrate responsibility and accountability?",
    "What should Monique do when Josh refuses to accept her apology?",
    "What can employees do to improve workplace morale?",
    "What should you do when your boss bad-mouths or belittles coworkers?",
    "What are the risks of dating a salesperson who supplies your office equipment?",
]

for q in test_questions:
    ask(q, verbose=True)

Pertanyaan: What does accountability mean in the workplace?

Chunks yang ditemukan (retrieval):
  [1] Halaman 2: 'Summary
Institut Teknologi Del 03
Employees need to be accountable,
but what does that really mean?
...'
  [2] Halaman 3: 'Summary
Institut Teknologi Del 03
Unfortunately, too many people do
not demonstrate accountability. ...'
  [3] Halaman 1: '1.Defining and linking concepts of empowerment,
personal responsibility, and accountability
2.Explai...'

Jawaban:
   Taking personal responsibility for ensuring work reflects positively on both you and your company. It includes aspects like project outcomes, customer experiences, and workplace and vendor relationships.
------------------------------------------------------------
Pertanyaan: Why should you avoid speaking poorly of others when networking?

Chunks yang ditemukan (retrieval):
  [1] Halaman 4: 'Talk it out
Institut Teknologi Del 03
1. Why should you not speak poorly of
others when networking?
...'
  [2] Halaman 4: 'nega

# 9. Gradio Chatbot UI
Tahap terakhir: membungkus pipeline RAG chain ke dalam antarmuka chatbot yang bisa diakses siapapun melalui browser.

**Cara kerja Gradio di Google Colab:**
- Gradio membuat **local server** di dalam Colab session
- Parameter `share=True` menghasilkan **public link** (tunnel ke server Gradio). Link aktif selama **72 jam** atau sampai session Google Colab ditutup

> Public link bisa dibagikan ke siapapun — tidak perlu login atau install apapun. Cocok untuk demo MVP yang langsung bisa dicoba.

### Penjelasan sel: Membuka tampilan chatbot

Sel ini membuat antarmuka Gradio agar chatbot bisa dicoba melalui kotak percakapan. Ubah `DOCUMENT_NAME` dan `example_questions` sesuai PDF yang aktif. Fungsi `chat_with_doc()` meneruskan pesan ke `rag_chain`. Parameter `history` belum digunakan untuk menyusun pertanyaan, sehingga model belum memakai riwayat percakapan sebagai konteks. Setelah dijalankan, buka tautan yang muncul. `share=True` membuat tautan yang dapat diakses orang lain selama server dan tautannya masih aktif. Catatan: label embedding pada tampilan masih menyebut `all-MiniLM-L6-v2`, sedangkan model dalam kode adalah `paraphrase-multilingual-MiniLM-L12-v2`.


In [16]:
import gradio as gr
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

DOCUMENT_NAME = "Accountability and Workplace Relationships"  # Ganti sesuai nama PDF

# Fungsi handler Gradio
# Dipanggil setiap kali user mengirim pesan
# history: list pesan sebelumnya (dikelola Gradio otomatis)
def chat_with_doc(message: str, history: list):
    if not message.strip():
        return "Silakan ketik pertanyaan Anda"
    try:
        return rag_chain.invoke(message)
    except Exception as e:
        return f"Error: {str(e)}"

# Ganti contoh pertanyaan sesuai isi PDF
example_questions = [
    ["What does accountability mean in the workplace?"],
    ["What can employees do to improve workplace morale?"],
    ["What should you do when your boss belittles coworkers?"],
]

# Bangun UI
with gr.Blocks(title=f"RAG Chatbot — {DOCUMENT_NAME}") as demo:

    gr.Markdown(f"""
    # RAG Chatbot — {DOCUMENT_NAME}
    Tanya apapun tentang dokumen ini!
    > **Tips:** Selalu gunakan pertanyaan spesifik untuk hasil terbaik.
    """)

    gr.ChatInterface(
        fn=chat_with_doc,
        chatbot=gr.Chatbot(
            height=450,
            allow_tags=False,
            placeholder="<strong>Halo! Saya siap menjawab pertanyaan Anda.</strong>",
            show_label=False,
        ),
        textbox=gr.Textbox(
            placeholder="Ketik pertanyaan Anda di sini...",
            container=False,
            scale=7
        ),
        examples=example_questions,
    )

    gr.Markdown(f"""
    ---
    **Model:** {MODEL_ID.split('/')[-1]} | **Embedding:** all-MiniLM-L6-v2 | **Vector DB:** FAISS
    """)

# Jalankan Gradio
# share=True  --> generate public link yang bisa dibagikan
# debug=False --> sembunyikan error internal dari user
# quiet=True  --> suppress log Gradio yang tidak perlu
demo.launch(share=True, debug=False, quiet=True)

* Running on public URL: https://32f7b1d2573919e4e2.gradio.live


### Penjelasan sel: Menutup chatbot jika sudah selesai

Sel ini menutup server Gradio. Jalankan hanya ketika chatbot sudah tidak dipakai. Lewati sel ini jika ingin antarmuka tetap terbuka. Jika terlanjur dijalankan, buka kembali chatbot dengan menjalankan ulang sel antarmuka pada langkah 9.


In [17]:
demo.close()

Closing server running on port: 7860


# 10. Opsional: Ganti Dokumen PDF tanpa Restart Session

Tahap ini melakukan re-indexing dokumen baru secara langsung. Model LLM tidak perlu di-load ulang karena model tetap sama, hanya vector database-nya yang diganti:
1. Upload PDF baru
2. Load & chunk dokumen baru
3. Buat embedding & FAISS index baru (menggantikan yang lama)
4. RAG chain otomatis terhubung ke dokumen baru


### Penjelasan sel: Opsional: mengganti PDF

Jalankan sel ini hanya jika ingin mengganti dokumen. Sel membaca PDF baru, memotong teks, membuat embedding, dan mengganti indeks FAISS beserta `chunks`. Model LLM tetap menggunakan model yang sudah dimuat. Dokumen baru menggantikan sumber pencarian sebelumnya, bukan menambahkannya. Setelah selesai, sesuaikan pertanyaan uji, nama dokumen, dan contoh pertanyaan di antarmuka. Output yang tersimpan pada versi ini menunjukkan penggantian ke paper tentang pengenalan emosi suara; karena itu, hasil uji Accountability di atas berasal dari dokumen sebelum penggantian.


In [ ]:
import numpy as np

# Upload PDF baru
print("Upload PDF baru...")
uploaded_new = files.upload()
new_pdf = list(uploaded_new.keys())[0]

# Load & ekstrak PDF baru
reader_new = PdfReader(new_pdf)
new_docs = []
for page_num, page in enumerate(reader_new.pages):
    text = page.extract_text()
    if text and text.strip():
        new_docs.append(Document(
            page_content=text,
            metadata={"page": page_num, "source": new_pdf}
        ))

# Chunk ulang
new_chunks = text_splitter.split_documents(new_docs)
print(f"{len(reader_new.pages)} halaman --> {len(new_chunks)} chunks")

# Re-embedding & rebuild FAISS index
# LLM tidak perlu di-load ulang. Hanya vector database yang diganti
new_texts = [c.page_content for c in new_chunks]
new_embeddings = embed_model.encode(
    new_texts,
    normalize_embeddings=True,
    show_progress_bar=True)

# Update variabel global index & chunks
# agar retrieve() otomatis pakai dokumen baru
dim = new_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(new_embeddings))
chunks = new_chunks

print(f"Berhasil ganti ke dokumen: {new_pdf}")
print(f"Total chunks baru: {index.ntotal}")

Upload PDF baru...


Saving 2025_Advancing_End-To-End_Indonesian_Speech_Emotion_Recognition_Using_XLSR-Wav2Vec2.pdf to 2025_Advancing_End-To-End_Indonesian_Speech_Emotion_Recognition_Using_XLSR-Wav2Vec2.pdf
6 halaman --> 69 chunks


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Berhasil ganti ke dokumen: 2025_Advancing_End-To-End_Indonesian_Speech_Emotion_Recognition_Using_XLSR-Wav2Vec2.pdf
Total chunks baru: 69
